# Alfa GPU worker

This notebook lends its GPU to an Alfa server. It asks the server for folding
jobs, runs them here, and uploads the results. **Nobody needs to watch it.**
If the session is torn down mid-job, the server notices within two minutes and
gives the job to the next worker; nothing is lost.

**One-time setup**

1. *Runtime → Change runtime type → T4 GPU.*
2. Open the 🔑 *Secrets* panel on the left and add:
   - `ALFA_SERVER`: the Alfa URL, e.g. `https://alfa.example.org`
   - `ALFA_WORKER_TOKEN`: from `pnpm worker:token "my colab" colab` on the server
   - Enable *Notebook access* for both.
3. *Runtime → Run all.*

In [ ]:
# Install the worker and ESMFold's model code. Colab already ships torch.
ALFA_REPO = "https://github.com/whywont/alfa-frontend"  # change if you forked it
!pip install -q "transformers>=4.44" accelerate "git+{ALFA_REPO}#subdirectory=worker"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import userdata

os.environ["ALFA_SERVER"] = userdata.get("ALFA_SERVER")
os.environ["ALFA_WORKER_TOKEN"] = userdata.get("ALFA_WORKER_TOKEN")
os.environ["ALFA_BACKEND"] = "colab"
os.environ["ALFA_WORKER_GIT_SHA"] = subprocess.run(
    ["git", "ls-remote", ALFA_REPO, "HEAD"], capture_output=True, text=True
).stdout.split()[0][:12] if ALFA_REPO else ""

In [ ]:
# Polls for work until the session ends. Stop it any time: a job in progress
# goes back to the queue when its lease expires.
!python -m alfa_worker --runner esmfold